### Installation

In [1]:
from google.colab import drive
drive.mount("/content/drive")

ModuleNotFoundError: No module named 'google'

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps git+https://github.com/huggingface/transformers.git
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

In [ ]:
%%capture
!pip install --no-deps --upgrade timm # For Gemma 4 vision/audio

### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None,
    max_seq_length = 2048,
    load_in_4bit = True,
    full_finetuning = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.1: Fast Gemma4 patching. Transformers: 5.6.0.dev0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

# Finetune Gemma 4

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 8,
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


<a name="Data"></a>
### Data Prep


In [ ]:
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)

# Make sure these match your actual filenames
TRAIN_PATH = "/content/drive/MyDrive/RADAI_Training_JSON/train.jsonl"
VAL_PATH   = "/content/drive/MyDrive/RADAI_Training_JSON/val.jsonl"

def load_json_or_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read().strip()
    if not text:
        return []
    if text[0] == "[":
        return json.loads(text)
    return [json.loads(line) for line in text.splitlines() if line.strip()]

def normalize_content(x):
    if x is None:
        return ""
    if isinstance(x, str):
        return x
    if isinstance(x, list):
        parts = []
        for item in x:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                if item.get("type") == "text":
                    parts.append(item.get("text", ""))
                elif "text" in item:
                    parts.append(item["text"])
        return "\n".join([p for p in parts if p])
    return str(x)

def normalize_role(role):
    role = str(role).lower()
    if role in ["assistant", "model", "gpt"]:
        return "assistant"
    if role in ["user", "human"]:
        return "user"
    if role == "system":
        return "system"
    return "user"

def to_messages(ex):
    # Case 1: already has messages
    if "messages" in ex:
        msgs = []
        for m in ex["messages"]:
            msgs.append({
                "role": normalize_role(m.get("role", "user")),
                "content": normalize_content(m.get("content", "")),
            })
        return {"messages": msgs}

    # Case 2: already has conversations
    if "conversations" in ex:
        msgs = []
        for m in ex["conversations"]:
            role = m.get("role", m.get("from", "user"))
            content = m.get("content", m.get("value", ""))
            msgs.append({
                "role": normalize_role(role),
                "content": normalize_content(content),
            })
        return {"messages": msgs}

    # Case 3: instruction style
    prompt = (
        ex.get("instruction")
        or ex.get("prompt")
        or ex.get("question")
        or ex.get("input")
        or ""
    )

    extra_input = ex.get("context") or ex.get("schema") or ""
    answer = (
        ex.get("output")
        or ex.get("response")
        or ex.get("answer")
        or ex.get("completion")
        or ""
    )

    if extra_input:
        prompt = f"{prompt}\n\n{extra_input}".strip()

    return {
        "messages": [
            {"role": "user", "content": str(prompt)},
            {"role": "assistant", "content": str(answer)},
        ]
    }

train_raw = load_json_or_jsonl(TRAIN_PATH)
val_raw   = load_json_or_jsonl(VAL_PATH)

train_ds = Dataset.from_list([to_messages(x) for x in train_raw])
val_ds   = Dataset.from_list([to_messages(x) for x in val_raw])

def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(
            msgs,
            tokenize = False,
            add_generation_prompt = False,
        )
        for msgs in examples["messages"]
    ]
    return {"text": texts}

train_ds = train_ds.map(formatting_prompts_func, batched=True)
val_ds   = val_ds.map(formatting_prompts_func, batched=True)

print(train_ds[0]["messages"])
print(train_ds[0]["text"][:1000])
print(val_ds)

Map:   0%|          | 0/81 [00:00<?, ? examples/s]

Map:   0%|          | 0/9 [00:00<?, ? examples/s]

[{'content': 'You are RAD-AI, a conservative radiation safety assistant. You do not invent measurements. If information is missing, ask clarifying questions. Provide actionable time-distance-shielding guidance.', 'role': 'system'}, {'content': 'Is it safe to start my shift?', 'role': 'user'}, {'content': "I can't determine whether it is safe to start your shift without more context.\n\nWhat I need to confirm:\n- What task will you be performing and how close to the source will you be working?\n- How long is your shift expected to last?\n- What is your site's administrative dose limit?", 'role': 'assistant'}]
<bos><|turn>user
You are RAD-AI, a conservative radiation safety assistant. You do not invent measurements. If information is missing, ask clarifying questions. Provide actionable time-distance-shielding guidance.

Is it safe to start my shift?<turn|>
<|turn>model
I can't determine whether it is safe to start your shift without more context.

What I need to confirm:
- What task wil

In [ ]:
print(train_raw[0])
print(train_ds[0]["messages"])

{'messages': [{'role': 'system', 'content': 'You are RAD-AI, a conservative radiation safety assistant. You do not invent measurements. If information is missing, ask clarifying questions. Provide actionable time-distance-shielding guidance.'}, {'role': 'user', 'content': 'Is it safe to start my shift?'}, {'role': 'assistant', 'content': "I can't determine whether it is safe to start your shift without more context.\n\nWhat I need to confirm:\n- What task will you be performing and how close to the source will you be working?\n- How long is your shift expected to last?\n- What is your site's administrative dose limit?"}]}
[{'content': 'You are RAD-AI, a conservative radiation safety assistant. You do not invent measurements. If information is missing, ask clarifying questions. Provide actionable time-distance-shielding guidance.', 'role': 'system'}, {'content': 'Is it safe to start my shift?', 'role': 'user'}, {'content': "I can't determine whether it is safe to start your shift withou

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,
        warmup_steps = 10,
        max_steps = 150,
        learning_rate = 2e-4,
        logging_steps = 5,
        eval_strategy = "steps",
        eval_steps = 25,
        save_strategy = "steps",
        save_steps = 75,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
        output_dir = "/content/gemma4_e2b_radai_test",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/81 [00:00<?, ? examples/s]

num_proc must be <= 9. Reducing num_proc to 9 for dataset of size 9.


Unsloth: Tokenizing ["text"] (num_proc=9):   0%|          | 0/9 [00:00<?, ? examples/s]

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)

Map (num_proc=16):   0%|          | 0/81 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/81 [00:00<?, ? examples/s]

num_proc must be <= 9. Reducing num_proc to 9 for dataset of size 9.


Map (num_proc=9):   0%|          | 0/9 [00:00<?, ? examples/s]

num_proc must be <= 9. Reducing num_proc to 9 for dataset of size 9.


Filter (num_proc=9):   0%|          | 0/9 [00:00<?, ? examples/s]

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

# Let's train the model!

To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 81 | Num Epochs = 14 | Total steps = 150
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 12,668,928 of 5,135,846,944 (0.25% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
25,4.701715,4.273893
50,2.996851,3.098584
75,2.044682,2.573286
100,1.407161,2.674023
125,0.648757,2.984868
150,0.513821,3.222294


In [ ]:
SAVE_PATH = "/content/drive/MyDrive/RADAI_Training_JSON/gemma4_e2b_radai_test_lora"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

['/content/drive/MyDrive/RADAI_Training_JSON/gemma4_e2b_radai_test_lora/processor_config.json']

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference


In [ ]:
messages = [{
    "role": "user",
    "content": [{"type": "text", "text": "You are RAD-AI. A user reports a dose rate of 120 uSv/h. Give a short, practical safety response."}]
}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 180,
    temperature = 1.0,
    top_p = 0.95,
    top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

At a dose rate of 120 µSv/h, this is slightly above the typical background for some areas, but still well within the regulatory limits for most public areas. However, it warrants awareness.

**Recommendation:** Inform the user that the dose rate is slightly elevated but generally acceptable. Advise them to remain aware of the area and take appropriate precautions if they are working or in a sensitive area.<turn|>


Merge model weights with LoRA adapter

In [ ]:
from unsloth import FastModel

MERGED_PATH = "/content/merged_gemma4_radai"

model.save_pretrained_merged(
    MERGED_PATH,
    tokenizer,
    save_method = "merged_16bit",
)

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/content/merged_gemma4_radai`: 100%|██████████| 1/1 [00:24<00:00, 24.29s/it]


Successfully copied all 1 files from cache to `/content/merged_gemma4_radai`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:02<00:00, 62.41s/it]


Unsloth: Merge process complete. Saved to `/content/merged_gemma4_radai`


<a name="Save"></a>
### convert into GGUF weights

In [ ]:
model.save_pretrained_gguf(
    "/content/gguf",
    tokenizer,
    quantization_method = "q4_k_m"
)

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/content/gguf`: 100%|██████████| 1/1 [00:27<00:00, 27.52s/it]


Successfully copied all 1 files from cache to `/content/gguf`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:06<00:00, 66.46s/it]


Unsloth: Merge process complete. Saved to `/content/gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/gguf_gguf/gemma-4-E2B-it.BF16.gguf', '/content/gguf_gguf/gemma-4-E2B-it.BF16-mmproj.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/gguf_gguf/gemma-4-E2B-it.Q4_K_M.gguf', '/content/gguf_gguf/gemma-4-E2B-it.BF16-mmproj.gguf']


Unsloth: example usage for Multimodal LLMs: /root/.unsloth/llama.cpp/llama-mtmd-cli -m /content/gguf_gguf/gemma-4-E2B-it.Q4_K_M.gguf --mmproj /content/gguf_gguf/gemma-4-E2B-it.BF16-mmproj.gguf
Unsloth: load image inside llama.cpp runner: /image test_image.jpg
Unsloth: Prompt model to describe the image
Unsloth: Saved Ollama Modelfile to /content/gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f /content/gguf_gguf/Modelfile


{'save_directory': '/content/gguf',
 'gguf_directory': '/content/gguf_gguf',
 'gguf_files': ['/content/gguf_gguf/gemma-4-E2B-it.Q4_K_M.gguf',
  '/content/gguf_gguf/gemma-4-E2B-it.BF16-mmproj.gguf'],
 'modelfile_location': '/content/gguf_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': True,
 'fix_bos_token': False}

download on PC

In [ ]:
from google.colab import files
files.download("/content/gguf_gguf/gemma-4-E2B-it.Q4_K_M.gguf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Saving to float16 for VLLM

We also support saving to `float16` directly for deployment! We save it in the folder `gemma-4-finetune`. Set `if False` to `if True` to let it run!

In [ ]:
if False: # Change to True to save finetune!
    model.save_pretrained_merged("gemma-4-finetune", tokenizer)

If you want to upload / push to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload finetune
    model.push_to_hub_merged(
        "HF_ACCOUNT/gemma-4-finetune", tokenizer,
        token = "YOUR_HF_TOKEN"
    )

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now for all models! For now, you can convert easily to `Q8_0, F16 or BF16` precision. `Q4_K_M` for 4bit will come later!

In [ ]:
if False: # Change to True to save to GGUF
    model.save_pretrained_gguf(
        "gemma_4_finetune",
        tokenizer,
        quantization_method = "Q8_0", # For now only Q8_0, BF16, F16 supported
    )

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload GGUF
    model.push_to_hub_gguf(
        "HF_ACCOUNT/gemma_4_finetune",
        tokenizer,
        quantization_method = "Q8_0", # Only Q8_0, BF16, F16 supported
        token = "YOUR_HF_TOKEN",
    )

Now, use the `gemma-4-finetune.gguf` file or `gemma-4-finetune-Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).